# Chat adapters: ChatAdapter, JSONAdapter, XMLAdapter, TwoStepAdapter, and a custom Markdown adapter

This notebook covers Chapter 7 section 7.4. Every time you call a DSPy module, an adapter translates your Signature into a prompt the model can understand. You usually never need to change it, but knowing how to swap or build one opens up XML, JSON, two-step extraction, and any custom format you prefer.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## 7.4.1 ChatAdapter

ChatAdapter is the default. It formats fields using bracket markers `[[ ## field_name ## ]]`. Use `dspy.inspect_history()` after any call to see the raw messages.


In [ ]:
import dspy

lm = dspy.LM("openai/gpt-5.6-luna")
dspy.configure(lm=lm)

recipe = dspy.Predict("ingredients: list[str] -> recipe_name: str, steps: str")
result = recipe(ingredients=["chicken", "lemon", "garlic", "rosemary"])

dspy.inspect_history(n=1)


## 7.4.2 JSONAdapter

DSPy does not select `JSONAdapter` automatically for a Pydantic output type, so configure it explicitly. It generates a JSON schema from your model, passes it to the API's structured output mode when supported, and returns a validated Python object.


In [ ]:
import dspy
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    title: str
    date: str
    time: str
    attendees: list[str]


dspy.configure(
    lm=lm,
    adapter=dspy.JSONAdapter()
)

extract_event = dspy.Predict(
    "email_text: str -> event: CalendarEvent"
)

# JSONAdapter parses the response into a validated Pydantic model
result = extract_event(
    email_text="Hey team, let's meet Thursday at 3pm to discuss the Q4 roadmap. "
               "Please confirm: Sarah, James, and Priya."
)
print(result.event.title)      # "Q4 Roadmap Discussion"
print(result.event.attendees)  # ["Sarah", "James", "Priya"]


The adapter remains configured for subsequent module calls until you replace it.


In [ ]:
assert isinstance(dspy.settings.adapter, dspy.JSONAdapter)


## 7.4.3 XMLAdapter

XMLAdapter swaps bracket markers for XML tags `<fieldname>content</fieldname>`. It extends ChatAdapter with minimal changes. Anthropic models historically preferred XML; the gap has narrowed but the format is much easier to read.

The inputs and outputs look like this when formatted with XMLAdapter:


```xml
<ingredients>
["chicken", "lemon", "garlic", "rosemary"]
</ingredients>
```


```xml
<recipe_name>
Lemon Rosemary Roast Chicken
</recipe_name>

<instructions>
Preheat oven to 425°F...
</instructions>
```


To enable XMLAdapter for all calls:


In [ ]:
dspy.configure(lm=lm, adapter=dspy.XMLAdapter())


## 7.4.4 TwoStepAdapter

Reasoning models like o3 and DeepSeek-R1 produce long chains of thought before their answer. `TwoStepAdapter` solves this with two LM calls: the main LM generates a natural-language response with no formatting constraints, then a smaller extraction LM parses the structured fields.


In [ ]:
import dspy

main_lm = dspy.LM("openai/gpt-5.6-sol", temperature=1.0, max_tokens=16000)
extraction_lm = dspy.LM("openai/gpt-5.6-luna")

adapter = dspy.TwoStepAdapter(extraction_model=extraction_lm)
dspy.configure(lm=main_lm, adapter=adapter)

analyze = dspy.ChainOfThought(
    "scenario: str -> recommendation: str, risk_level: str, confidence: float"
)

result = analyze(
    scenario="A Series B startup wants to pivot from B2B SaaS to consumer mobile"
)


## 7.4.5 Building a custom MarkdownAdapter

The adapter hierarchy gives you two entry points. Subclass `ChatAdapter` to override how fields are formatted and parsed; subclass `Adapter` directly for full control over the message structure. Here's a `MarkdownAdapter` that formats fields as Markdown headings.


In [ ]:
import re
from typing import Any
from dspy.adapters.chat_adapter import ChatAdapter, FieldInfoWithName
from dspy.adapters.utils import format_field_value, parse_value
from dspy.signatures.signature import Signature

class MarkdownAdapter(ChatAdapter):
    """Formats DSPy fields as Markdown headings instead of bracket markers."""

    def __init__(self):
        super().__init__()
        self.field_pattern = re.compile(
            r"^## (?P<name>\w+)\n(?P<content>.*?)(?=^## |\Z)",
            re.MULTILINE | re.DOTALL
        )

    def format_field_with_value(self, fields_with_values: dict[FieldInfoWithName, Any]) -> str:
        """Format each field as a Markdown heading with content below."""
        output = []
        for field, field_value in fields_with_values.items():
            formatted = format_field_value(field_info=field.info, value=field_value)
            output.append(f"## {field.name}\n{formatted}")
        return "\n\n".join(output).strip()

    def user_message_output_requirements(self, signature: type[Signature]) -> str:
        """Tell the model to respond in Markdown heading format."""
        fields = ", ".join(f"`## {f}`" for f in signature.output_fields)
        return f"Respond using Markdown headings for: {fields}, then end with `## completed`."

    def parse(self, signature: type[Signature], completion: str) -> dict[str, Any]:
        """Extract fields from Markdown-formatted response."""
        fields = {}
        for match in self.field_pattern.finditer(completion):
            name = match.group("name").strip()
            content = match.group("content").strip()
            if name in signature.output_fields and name not in fields:
                fields[name] = parse_value(content, signature.output_fields[name].annotation)
        return fields


Use it like any other adapter.


In [ ]:
dspy.configure(lm=lm, adapter=MarkdownAdapter())
